In [1]:
import random
from rdkit import Chem
from molpher.core import MolpherMol, MolpherAtom
from molpher.core.morphing.operators import MorphingOperator
from rdkit.Chem.EnumerateStereoisomers import EnumerateStereoisomers, StereoEnumerationOptions
from rdkit.Chem import rdChemReactions
from rdkit.Chem import rdmolops
from rdkit.Chem import Descriptors  
from molpher.core import ExplorationTree as ETree

class AromaticHydroxylation(MorphingOperator):
    def __init__(self):
        super(AromaticHydroxylation, self).__init__()
        self._name = "Aromatic Hydroxylation (Phase I - Regioselective)"
        self._matches = []
        self.AROMATIC_C = Chem.MolFromSmarts("[c;H1]")

    def _get_para_score(self, mol, c_idx):
        """ Υπολογισμός para-θέσης σε 6μελείς δακτυλίους """
        for ring in mol.GetRingInfo().AtomRings():
            if c_idx in ring and len(ring) == 6:
                for r_idx in ring:
                    r_atom = mol.GetAtomWithIdx(r_idx)
                    has_ex_neighbor = any(n.GetIdx() not in ring for n in r_atom.GetNeighbors())
                    
                    if has_ex_neighbor:
                        path = Chem.GetShortestPath(mol, r_idx, c_idx)
                        if len(path) == 4: # Απόσταση para
                            return 10
        return 1 

    def setOriginal(self, mol):
        super(AromaticHydroxylation, self).setOriginal(mol)
        self._matches = []
        
        if not self.original: return
        rdkit_mol = self.original.asRDMol()
        if rdkit_mol is None: return

        if self.AROMATIC_C is not None:
            matches = rdkit_mol.GetSubstructMatches(self.AROMATIC_C)
            scored_sites = []
            for match in matches:
                c_idx = match[0]
                score = self._get_para_score(rdkit_mol, c_idx)
                scored_sites.append((c_idx, score))
                
            if scored_sites:
                max_score = max(site[1] for site in scored_sites)
                self._matches = [site[0] for site in scored_sites if site[1] == max_score]

    def morph(self):
        if not self.original: return None
        rdkit_mol = self.original.asRDMol()
        if rdkit_mol is None: return None

        if not self._matches:
            return MolpherMol(other=rdkit_mol)

        c_idx = random.choice(self._matches)
        
        try:
            rw_mol = Chem.RWMol(rdkit_mol)
            
            new_o_idx = rw_mol.AddAtom(Chem.Atom(8))
            c_atom = rw_mol.GetAtomWithIdx(c_idx)
            o_atom = rw_mol.GetAtomWithIdx(new_o_idx)
            
            c_atom.SetNoImplicit(False)
            c_atom.SetNumExplicitHs(0)
            o_atom.SetNoImplicit(False)
            o_atom.SetNumExplicitHs(0)
            
            rw_mol.AddBond(c_idx, new_o_idx, Chem.BondType.SINGLE)
            
            new_mol = rw_mol.GetMol()
            
            new_mol.UpdatePropertyCache(strict=False)
            Chem.SanitizeMol(new_mol, Chem.SanitizeFlags.SANITIZE_ALL)
            Chem.AssignStereochemistry(new_mol, cleanIt=True, force=True)
            
            return MolpherMol(other=new_mol)
        except:
            return MolpherMol(other=rdkit_mol)

    def getName(self): return self._name

class OMethylation(MorphingOperator):
    def __init__(self):
        super(OMethylation, self).__init__()
        self._name = "O-methylation (Alcohols/Phenols)"
        self._target_atoms = []
        self.PATTERN = Chem.MolFromSmarts("[OX2H][#6;!$(C=O);!$(C=C);!$(C=N)]")

    def setOriginal(self, mol):
        super(OMethylation, self).setOriginal(mol)
        self._target_atoms = []
        if not self.original: return
        rdkit_mol = self.original.asRDMol()
        if rdkit_mol is None: return
        
        matches = rdkit_mol.GetSubstructMatches(self.PATTERN)
        for match in matches:
            self._target_atoms.append(match[0])

    def morph(self):
        if not self.original or not self._target_atoms:
            return MolpherMol(other=self.original.asRDMol())
        
        rdkit_mol = self.original.asRDMol()
        try:
            target_o_idx = random.choice(self._target_atoms)
            rw_mol = Chem.RWMol(rdkit_mol)
            
            c_methyl_idx = rw_mol.AddAtom(Chem.Atom(6))
            rw_mol.AddBond(target_o_idx, c_methyl_idx, Chem.BondType.SINGLE)
            
            new_mol = rw_mol.GetMol()
            
            for idx in [target_o_idx, c_methyl_idx]:
                atom = new_mol.GetAtomWithIdx(idx)
                atom.SetNoImplicit(False)
                atom.SetNumExplicitHs(0)
                atom.SetFormalCharge(0)
                
            new_mol.UpdatePropertyCache(strict=False)
            Chem.SanitizeMol(new_mol, Chem.SanitizeFlags.SANITIZE_ALL)
            Chem.AssignStereochemistry(new_mol, cleanIt=True, force=True)
            
            return MolpherMol(other=new_mol)
            
        except:
            return MolpherMol(other=rdkit_mol)

    def getName(self): return self._name

hydroxylation_op = AromaticHydroxylation()
omethylation_op = OMethylation() 

class FindClosest:
    def __init__(self):
        self.closest_mol = None
        self.closest_distance = None
        
    def __call__(self, morph):
        if not self.closest_mol or self.closest_distance > morph.dist_to_target:
            self.closest_mol = morph
            self.closest_distance = morph.dist_to_target
closest_info = FindClosest()
start_mol  = MolpherMol("NCCc1ccc(O)cc1")
target_mol = MolpherMol("NCCc1ccc(O)c(OC)c1") 
tree = ETree.create(source=start_mol, target=target_mol)
tree.morphing_operators = (hydroxylation_op, omethylation_op)

print("--- STARTING MOLPHER SEARCH TREE ---")
max_generations = 40
while not tree.path_found and tree.generation_count < max_generations:
    tree.generateMorphs()
    tree.sortMorphs()
    tree.filterMorphs()
    tree.extend()
    tree.prune()
    tree.traverse(closest_info)
    
    print(f"Generation #{tree.generation_count}")
    print(f"Molecules in tree: {tree.mol_count}")
    if closest_info.closest_mol:
        print(f"Closest to target: {closest_info.closest_mol.getSMILES()} (Distance: {closest_info.closest_distance:.4f})")
    print("-" * 40)


--- STARTING MOLPHER SEARCH TREE ---
Generation #1
Molecules in tree: 6
Closest to target: NCCC1=CC=C(O)C(O)=C1 (Distance: 0.4000)
----------------------------------------
Generation #2
Molecules in tree: 16
Closest to target: COC1=CC(CCN)=CC=C1O (Distance: 0.0000)
----------------------------------------
